# Shared-core smoke test (archival)

## Deterministic shared-core validation notebook

This notebook is part of the **canonical archival chain** in `config/notebook_plan.json`. It shows that the repository's checked-out **corrected public engine v1.1** runs end-to-end on a **single fixed synthetic case** and writes the two artefacts required for cryptographic validation.

### Domain terms (what is being evaluated)

The engine scores a candidate AI-governance **case** using five **non-compensatory gates** (safety, evidence, bias or equity, calibration, traceability) and a **compensatory** score built from the same five feature dimensions with canonical weights. In **`replay_mode`**, the final approve/reject flag follows the replay rule documented in the engine: all gates must pass for a non-reject path at the gate layer, and SCM-style abstention overrides are **not** applied. This matches the static historical replay baseline the codebase documents.


## Outputs this notebook owns

Per `config/trace_map.json` and `config/expected_outputs.json`, **only** these paths may be produced here:

- `outputs/tables/smoke_test_results.csv` — one-row evaluation record plus a SHA-256 digest of the canonical JSON result object.
- `outputs/figures/smoke_test_summary.txt` — human-readable mirror of the same run for reviewers and auditors.

No other files are written. Execution is **deterministic** (fixed case dict, fixed **moderate** profile, fixed `replay_mode`) so byte-level hashes stay stable for a locked release.


## Engine entry points used

- `evaluate_case` — full case evaluation for a named threshold profile.
- `MODE_REPLAY` — evaluation without the SCM abstention / fallback layer in the final verdict.
- `hash_output` — SHA-256 over sorted-keys JSON of the result dict (embedded in the CSV and echoed in the summary).

Implementation: `engine/corrected_public_engine_v1_1.py` (imported below, not reimplemented).


In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

OUT_TAB = ROOT / "outputs" / "tables"
OUT_FIG = ROOT / "outputs" / "figures"
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

CASE = {
    "case_id": "smoke_core_001",
    "features": {
        "intrinsic_safety": 0.58,
        "evidence_strength": 0.55,
        "bias_harm_index": 0.44,
        "uncertainty_calibration": 0.52,
        "traceability_integrity": 0.53,
    },
}



## Evaluation step

The next cell calls `evaluate_case` on `CASE` using the **moderate** threshold profile from `CANONICAL_THRESHOLD_PROFILES` in the engine module. The returned dict includes per-gate pass bits, compensatory score versus threshold, compensatory approval, and governance outcome strings. Changing the case dict or profile would change the locked artefacts and break `config/expected_outputs.json` pins.


In [2]:
result = eng.evaluate_case(CASE, profile_name="moderate", mode=eng.MODE_REPLAY)
result_hash = eng.hash_output(result)


## Write declared artefacts

The CSV layout and summary lines below are the **exact** serialisation expected by the release manifest. Do not edit wording or column order without updating hash pins and re-validating the full harness.


In [3]:
csv_path = OUT_TAB / "smoke_test_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "case_id",
            "profile_name",
            "mode",
            "governance_outcome",
            "all_gates_pass",
            "compensatory_score",
            "compensatory_approved",
            "approved",
            "result_sha256",
        ]
    )
    w.writerow(
        [
            result["case_id"],
            result["profile_name"],
            result["mode"],
            result["governance_outcome"],
            result["all_gates_pass"],
            result["compensatory_score"],
            result["compensatory_approved"],
            result["approved"],
            result_hash,
        ]
    )



In [4]:
summary_path = OUT_FIG / "smoke_test_summary.txt"
lines = [
    "Shared-core smoke test (01_smoke_test)",
    "========================================",
    "",
    "Purpose: This notebook runs one fixed synthetic governance case through the",
    "corrected public engine v1.1 in replay_mode (five gates plus compensatory",
    "scoring only, matching the static historical replay path).",
    "",
    "Why this matters: A successful run proves the checked-out engine module",
    "executes end-to-end and that we can serialize a deterministic record under",
    "config/expected_outputs.json for automated manifest validation.",
    "",
    "Outputs produced (this notebook only):",
    "- outputs/tables/smoke_test_results.csv — tabular record for the case.",
    "- outputs/figures/smoke_test_summary.txt — this narrative summary.",
    "",
    f"Governance outcome:                  {result['governance_outcome']}",
    f"All gates pass (replay):             {result['all_gates_pass']}",
    f"Compensatory approved:               {result['compensatory_approved']}",
    f"Final approved flag (replay):       {result['approved']}",
    f"SHA-256 (canonical JSON result):    {result_hash}",
    "",
]
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Smoke test artifacts written.")



Smoke test artifacts written.


## Interpreting the result (tied to the code above)

- **`governance_outcome`** and **`approved`** summarise the replay verdict for the fixed case. They are produced by the engine, not invented in this notebook.
- **`result_sha256`** fingerprints the full result object; the harness uses the same hashing primitive to detect any drift in serialised outputs.
- **Why auditors care:** a passing `python reproduce_all.py` run with hash-locked `expected_outputs.json` shows this notebook still produces the **exact** bytes promised for the shared core, with no manual steps.

This completes the deterministic archival smoke path.
